In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import requests
from io import StringIO

In [2]:
url = "https://raw.githubusercontent.com/junaart/AHP_KIS/main/Lab_1/computer_dataset_1000.csv"

try:
    df = pd.read_csv(url, sep=',', encoding='utf-8', skipinitialspace=True)
    df = df.dropna(how='all')
    df = df.reset_index(drop=True)
    print(f"Данные успешно загружены. Всего строк: {len(df)}")
except Exception as e:
    print(f"Ошибка при чтении pandas: {e}")
    response = requests.get(url)
    response.encoding = 'utf-8'
    df = pd.read_csv(StringIO(response.text), sep=',', encoding='utf-8', skipinitialspace=True)
    df = df.dropna(how='all')
    df = df.reset_index(drop=True)
    print(f"Данные загружены. Всего строк: {len(df)}")

total_rows = len(df)


Данные успешно загружены. Всего строк: 1000


In [ ]:
info_label = widgets.HTML(
    value=f"<b>Всего записей: {total_rows}</b>",
    layout=widgets.Layout(margin='0 0 5px 0')
)

row_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=total_rows - 1,
    step=1,
    description='Строка:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%', margin='5px 0')
)



row_number = widgets.BoundedIntText(
    value=1,
    min=1,
    max=total_rows,
    step=1,
    description='Номер строки:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='28%', margin='5px 0')
)

rows_count = widgets.Dropdown(
    options=[('1 строка', 1), ('3 строки', 3), ('5 строк', 5)],
    value=1,
    description='Показывать:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px', margin='5px 0')
)

btn_first = widgets.Button(
    description='Первая',
    button_style='info',
    layout=widgets.Layout(width='100px', margin='5px 2px')
)

btn_prev = widgets.Button(
    description='Предыдущая',
    button_style='primary',
    layout=widgets.Layout(width='100px', margin='5px 2px')
)

btn_next = widgets.Button(
    description='Следующая',
    button_style='primary',
    layout=widgets.Layout(width='100px', margin='5px 2px')
)

btn_last = widgets.Button(
    description='Последняя',
    button_style='info',
    layout=widgets.Layout(width='100px', margin='5px 2px')
)

status_label = widgets.HTML(
    value="",
    layout=widgets.Layout(margin='5px 0 0 0')
)

output_area = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ddd',
        padding='10px',
        max_height='600px',
        overflow_y='auto'
    )
)

In [4]:
def create_html_table(subset_df, start_idx):
    """Создает красивую HTML-таблицу из DataFrame"""
    
    # CSS стили для таблицы
    table_style = """
    <style>
        .data-table {
            width: 100%;
            border-collapse: collapse;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
            font-size: 13px;
            background-color: white;
        }
        
        .data-table th {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 12px 8px;
            text-align: left;
            font-weight: 600;
            position: sticky;
            top: 0;
            z-index: 10;
            border: 1px solid #5a67d8;
        }
        
        .data-table td {
            padding: 8px;
            border: 1px solid #e2e8f0;
            color: #2d3748;
        }
        
        .data-table tr:hover {
            background-color: #f7fafc;
            transition: background-color 0.2s ease;
        }
        
        .data-table tr:nth-child(even) {
            background-color: #f9fafb;
        }
        
        .row-header {
            background-color: #edf2f7;
            font-weight: 600;
        }
        
        .table-container {
            max-height: 500px;
            overflow: auto;
            border-radius: 8px;
            box-shadow: 0 1px 3px 0 rgba(0,0,0,0.1);
        }
        
        .row-number {
            background-color: #667eea;
            color: white;
            font-weight: 600;
            text-align: center;
            width: 60px;
        }
    </style>
    """
    
    # Создаем HTML-таблицу
    html = '<div class="table-container">'
    html += '<table class="data-table">'
    
    # Заголовки
    html += '<thead><tr>'
    html += '<th class="row-number">№</th>'
    for col in subset_df.columns:
        html += f'<th>{col}</th>'
    html += '</tr></thead>'
    
    # Данные
    html += '<tbody>'
    for idx, (_, row) in enumerate(subset_df.iterrows()):
        actual_row_num = start_idx + idx + 1
        html += '<tr>'
        html += f'<td class="row-number" style="background-color: #667eea; color: white; text-align: center; font-weight: 600;">{actual_row_num}</td>'
        for col in subset_df.columns:
            value = row[col]
            if pd.isna(value) or str(value).strip() == '':
                value = '-'
            html += f'<td>{value}</td>'
        html += '</tr>'
    html += '</tbody></table></div>'
    
    return table_style + html

def display_formatted_table(start_idx, n_rows):
    """Отображает строки в виде красивой HTML-таблицы"""
    with output_area:
        output_area.clear_output()
        
        end_idx = min(start_idx + n_rows, total_rows)
        subset = df.iloc[start_idx:end_idx]
        
        # Создаем и отображаем HTML-таблицу
        html_table = create_html_table(subset, start_idx)
        display(HTML(html_table))
        
        # Обновляем статус
        if n_rows == 1:
            status_label.value = f"<span style='color: #2c5282;'>Отображена строка {start_idx+1} из {total_rows}</span>"
        else:
            status_label.value = f"<span style='color: #2c5282;'>Отображены строки {start_idx+1}-{end_idx} из {total_rows}</span>"

def update_display(*args):
    """Обновляет отображение на основе текущих значений виджетов"""
    current_idx = row_slider.value
    
    if row_number.value != current_idx + 1:
        row_number.value = current_idx + 1
    
    n = rows_count.value
    start_idx = current_idx
    
    if start_idx + n > total_rows:
        start_idx = total_rows - n
        if start_idx < 0:
            start_idx = 0
        if row_slider.value != start_idx:
            row_slider.value = start_idx
            return
    
    display_formatted_table(start_idx, n)

In [5]:
def sync_from_slider(change):
    row_number.value = change['new'] + 1
    update_display()

def sync_from_number(change):
    new_idx = change['new'] - 1
    if 0 <= new_idx < total_rows:
        row_slider.value = new_idx

def change_rows_per_page(change):
    update_display()

def go_first(b):
    row_slider.value = 0

def go_last(b):
    max_start = total_rows - rows_count.value
    if max_start < 0:
        max_start = 0
    row_slider.value = max_start

def go_prev(b):
    new_idx = row_slider.value - rows_count.value
    if new_idx < 0:
        new_idx = 0
    row_slider.value = new_idx

def go_next(b):
    new_idx = row_slider.value + rows_count.value
    max_start = total_rows - rows_count.value
    if max_start < 0:
        max_start = 0
    if new_idx > max_start:
        new_idx = max_start
    row_slider.value = new_idx

In [6]:
row_slider.observe(sync_from_slider, names='value')
row_number.observe(sync_from_number, names='value')
rows_count.observe(change_rows_per_page, names='value')

btn_first.on_click(go_first)
btn_prev.on_click(go_prev)
btn_next.on_click(go_next)
btn_last.on_click(go_last)

In [7]:
controls_panel = widgets.VBox([
    info_label,
    widgets.HBox([row_slider, row_number]),
    widgets.HBox([rows_count, btn_first, btn_prev, btn_next, btn_last]),
    status_label,
    widgets.HTML("<hr style='margin: 5px 0; border-color: #e2e8f0;'>")
], layout=widgets.Layout(margin='10px'))

display(controls_panel, output_area)

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

In [8]:

update_display()

print("\nИнформация о данных:")
print(f"  - Всего строк: {total_rows}")
print(f"  - Всего колонок: {len(df.columns)}")


Информация о данных:
  - Всего строк: 1000
  - Всего колонок: 31
